# Graph-Flashback on Gowalla — Kaggle

Notebook клонирует `flashback-branch`, проверяет GPU, выполняет smoke-test, затем запускает полный pipeline по стадиям. В Kaggle включите **Internet** и **GPU T4**.

In [ ]:
%cd /kaggle/working
!rm -rf GeoGNNProject
!git clone --depth 1 --branch flashback-branch --single-branch https://github.com/klyuchnikova/GeoGNNProject.git
%cd /kaggle/working/GeoGNNProject
!git branch --show-current
!git log -1 --oneline
!python scripts/verify_merge.py


## Dependencies and GPU

In [ ]:
!grep -v '^torch' requirements.txt > /tmp/requirements-kaggle.txt
!python -m pip install -q -r /tmp/requirements-kaggle.txt

import torch, pandas, numpy, scipy
print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
print("Pandas:", pandas.__version__)
print("NumPy:", numpy.__version__)
print("SciPy:", scipy.__version__)


## Smoke-test

In [ ]:
!python scripts/make_synthetic_data.py
!python -m flashback.pipeline --config configs/gowalla_smoke.yaml --stage all
!python -m pytest -q --basetemp=/kaggle/working/pytest_tmp


## Full Gowalla development run (70/10/20)

Run the cells one by one. Completed stages are saved on disk.

In [ ]:
CONFIG = "configs/gowalla_auto.yaml"

In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage download


In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage prepare


In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage stkg


In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage kge


In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage graphs


In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage train


In [ ]:
!python -m flashback.pipeline --config $CONFIG --stage analyze


## Inspect results

In [ ]:
from pathlib import Path
import json, pandas as pd

for path in sorted(Path("artifacts/results").glob("*.json")):
    print("\n", path)
    print(json.dumps(json.loads(path.read_text()), indent=2))

ranking = Path("data/processed/city_ranking.csv")
if ranking.exists():
    display(pd.read_csv(ranking))


## Package lightweight results for review

In [ ]:
from pathlib import Path
import zipfile

root = Path("/kaggle/working/GeoGNNProject")
out = Path("/kaggle/working/graph_flashback_results.zip")
include = [root / "artifacts", root / "data/processed", root / "data/kge/stkg_manifest.json", root / "data/kge/transe_history.json", root / "data/graphs/graph_manifest.json", root / "configs"]
excluded = {".pt", ".pth", ".pkl", ".npz", ".npy"}
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as archive:
    for path in include:
        if not path.exists():
            continue
        files = [path] if path.is_file() else path.rglob("*")
        for file in files:
            if file.is_file() and file.suffix.lower() not in excluded:
                archive.write(file, file.relative_to(root))
print(out, f"{out.stat().st_size / 1024**2:.2f} MB")
